In [5]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = "/content/"
DATA_CSV    = os.path.join(BASE_DIR, "processed_data.csv")
ENCODER_PKL = os.path.join(BASE_DIR, "station_encoder.pkl")
PLOT_DIR    = "/content/plots"

if os.path.exists(PLOT_DIR) and os.path.isfile(PLOT_DIR):
    os.remove(PLOT_DIR)
os.makedirs(PLOT_DIR, exist_ok=True)

FEATURE_COLS = [
    "station_encoded", "destination_encoded", "arrival_delay",
    "planned_arr_mins", "day_of_week", "week_of_year", "is_peak_hour",
]
TARGET_COL = "target_delay_at_destination"

PALETTE = ["#20808D", "#A84B2F", "#1B474D", "#944454", "#FFC553", "#848456"]

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#E0DDD8",
    "grid.linewidth": 0.6,
})

# ── 1. Load data ───────────────────────────────────────────────────────────────
print("=== Loading dataset ===")
df = pd.read_csv(DATA_CSV)
with open(ENCODER_PKL, "rb") as f:
    encoder = pickle.load(f)

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
groups = df["journey_id"].values
print(f"  {len(df):,} rows | {df['journey_id'].nunique():,} journeys")

# ── 2. Train/test split (by journey) ──────────────────────────────────────────
print("\n=== Train/Test Split ===")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
g_train         = groups[train_idx]
print(f"  Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# Cross-val splitter (group-aware)
cv = GroupKFold(n_splits=3)

# ── 3. Evaluate helper ─────────────────────────────────────────────────────────
def evaluate(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return dict(
        RMSE     = float(np.sqrt(mean_squared_error(y_te, y_pred))),
        MAE      = float(mean_absolute_error(y_te, y_pred)),
        R2       = float(r2_score(y_te, y_pred)),
        Within2  = float(np.mean(np.abs(y_pred - y_te) <= 2)  * 100),
        Within5  = float(np.mean(np.abs(y_pred - y_te) <= 5)  * 100),
        Within10 = float(np.mean(np.abs(y_pred - y_te) <= 10) * 100),
        y_pred   = y_pred,
    )

# ── 4. Grid search configs ─────────────────────────────────────────────────────
print("\n=== GridSearchCV (group-aware, 3-fold) ===")

search_configs = {
    "Ridge": {
        "estimator": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
        "param_grid": {
            "model__alpha": [0.001, 0.01],
            "model__solver": ["auto"],
        },
        "n_jobs": -1,
    },
    "Random Forest": {
        "estimator": RandomForestRegressor(random_state=42, n_jobs=1),
        "param_grid": {
            "n_estimators": [100],
            "max_depth":    [20],
            "max_features": [0.5],
        },
        "n_jobs": 1,   # prevent OOM
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingRegressor(random_state=42),
        "param_grid": {
            "n_estimators":  [200],
            "max_depth":     [5],
            "learning_rate": [0.05],
        },
        "n_jobs": -1,
    },
    "XGBoost": {
        "estimator": xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
        "param_grid": {

            "max_depth":        [8],
            "learning_rate":    [0.1],
            "subsample":        [0.8],
        },
        "n_jobs": -1,
    },
    "LightGBM": {
        "estimator": lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
        "param_grid": {
            "learning_rate": [0.1],
            "num_leaves":    [127],
            "subsample":     [0.8],
        },
        "n_jobs": -1,
    },
}

tuned_models  = {}
tuned_results = {}
best_params   = {}
all_cv_records = []   # will hold every grid point for every model

for name, cfg in search_configs.items():
    param_grid = cfg["param_grid"]
    n_combos = 1
    for v in param_grid.values():
        n_combos *= len(v)
    print(f"\n  {name}  ({n_combos} combinations × 3-fold CV) …")

    search = GridSearchCV(
        estimator  = cfg["estimator"],
        param_grid = param_grid,
        cv         = cv,
        scoring    = "neg_mean_absolute_error",
        n_jobs     = cfg["n_jobs"],
        verbose    = 0,
        refit      = True,
        return_train_score = False,
    )
    search.fit(X_train, y_train, groups=g_train)

    best_params[name] = search.best_params_
    print(f"    Best params : {search.best_params_}")
    print(f"    Best CV MAE : {-search.best_score_:.4f}")

    # ── Collect all CV results for this model ──────────────────────────────
    cv_df = pd.DataFrame(search.cv_results_)
    cv_df.insert(0, "model", name)
    # rename score columns to be human-readable
    cv_df = cv_df.rename(columns={
        "mean_test_score": "cv_mean_neg_mae",
        "std_test_score":  "cv_std_neg_mae",
        "rank_test_score": "cv_rank",
    })
    # add positive MAE for readability
    cv_df["cv_mean_mae"] = -cv_df["cv_mean_neg_mae"]
    cv_df["cv_std_mae"]  =  cv_df["cv_std_neg_mae"]
    all_cv_records.append(cv_df)

    # ── Evaluate best model on held-out test set ───────────────────────────
    best_model = search.best_estimator_
    tuned_models[name]  = best_model
    res = evaluate(best_model, X_test, y_test)
    tuned_results[name] = res
    print(f"    Test  MAE={res['MAE']:.3f}  RMSE={res['RMSE']:.3f}  R²={res['R2']:.4f}")

# ── 5. Save all CV results to CSV ─────────────────────────────────────────────
print("\n=== Saving CV results ===")
all_cv_df = pd.concat(all_cv_records, ignore_index=True)

# Keep the most useful columns; drop per-fold split times etc.
keep_cols = (
    ["model", "cv_rank", "cv_mean_mae", "cv_std_mae"]
    + [c for c in all_cv_df.columns if c.startswith("param_")]
    + ["mean_fit_time", "mean_score_time"]
)
keep_cols = [c for c in keep_cols if c in all_cv_df.columns]
all_cv_df = all_cv_df[keep_cols].sort_values(["model", "cv_rank"])

cv_csv_path = os.path.join(BASE_DIR, "grid_search_cv_results.csv")
all_cv_df.to_csv(cv_csv_path, index=False)
print(f"  ✓ {len(all_cv_df)} rows saved → {cv_csv_path}")

# Save best params JSON
with open(os.path.join(BASE_DIR, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2, default=str)
print(f"  ✓ best_params.json saved → {BASE_DIR}")

# ── 6. Plots ───────────────────────────────────────────────────────────────────
model_names = list(tuned_results.keys())
x = np.arange(len(model_names))

# ── Plot A: MAE & RMSE bar chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Grid Search — Tuned Model Performance\n(Waterloo ↔ Weymouth, 2022–2025)",
             fontsize=14, fontweight="bold", y=1.02)

for ax, metric, title, ylabel in [
    (axes[0], "MAE",  "Mean Absolute Error (MAE)",      "minutes"),
    (axes[1], "RMSE", "Root Mean Squared Error (RMSE)", "minutes"),
]:
    vals = [tuned_results[n][metric] for n in model_names]
    bars = ax.bar(model_names, vals, color=PALETTE[:len(model_names)],
                  edgecolor="white", width=0.55, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=9, color="#28251D")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0, max(vals) * 1.18)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/07_tuned_mae_rmse.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✓ saved 07_tuned_mae_rmse.png")

# ── Plot B: R² ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
r2_vals = [tuned_results[n]["R2"] for n in model_names]
bars = ax.bar(model_names, r2_vals, color=PALETTE[:len(model_names)],
              edgecolor="white", width=0.55, alpha=0.9)
for bar, v in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002,
            f"{v:.4f}", ha="center", va="bottom", fontsize=9, color="#28251D")
ax.axhline(0, color="#28251D", lw=0.8, linestyle="--")
ax.set_ylabel("R²")
ax.set_title("R² Score — Tuned Models (Grid Search)")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/08_tuned_r2.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 08_tuned_r2.png")

# ── Plot C: Within-N-minutes accuracy ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
width = 0.25
for i, (tol, col) in enumerate([(2, PALETTE[0]), (5, PALETTE[1]), (10, PALETTE[2])]):
    vals = [tuned_results[n][f"Within{tol}"] for n in model_names]
    bars = ax.bar(x + (i - 1) * width, vals, width,
                  label=f"Within {tol} min", color=col, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f"{v:.0f}%", ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylabel("% of Predictions")
ax.set_title("Within-N-Minutes Accuracy — Tuned Models")
ax.legend(loc="lower right", fontsize=9)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/09_tuned_within_n.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 09_tuned_within_n.png")

# ── Plot D: Actual vs Predicted ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Actual vs Predicted — Tuned Models (Grid Search)",
             fontsize=14, fontweight="bold")
axes = axes.flatten()
sample = np.random.RandomState(42).choice(len(y_test),
         size=min(5000, len(y_test)), replace=False)
lim_lo, lim_hi = -15, 45

for ax, (name, col) in zip(axes, zip(model_names, PALETTE)):
    y_pred = tuned_results[name]["y_pred"]
    ax.scatter(y_test[sample], y_pred[sample],
               alpha=0.25, s=4, color=col, rasterized=True)
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", lw=1.2, label="Perfect")
    ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
    ax.set_xlabel("Actual (min)"); ax.set_ylabel("Predicted (min)")
    ax.set_title(f"{name}\nR²={tuned_results[name]['R2']:.3f}  "
                 f"MAE={tuned_results[name]['MAE']:.2f} min")
    ax.legend(fontsize=8)

# hide unused subplot if any
for ax in axes[len(model_names):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/10_tuned_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 10_tuned_actual_vs_predicted.png")

# ── Plot E: Performance heatmap ────────────────────────────────────────────────
metrics_display = ["RMSE\n(min)", "MAE\n(min)", "R²",
                   "Within\n2 min (%)", "Within\n5 min (%)", "Within\n10 min (%)"]
metric_keys = ["RMSE", "MAE", "R2", "Within2", "Within5", "Within10"]

raw_values = np.array([
    [tuned_results[n][k] for k in metric_keys]
    for n in model_names
])
norm = raw_values.copy().astype(float)
for col_i, key in enumerate(metric_keys):
    lo, hi = norm[:, col_i].min(), norm[:, col_i].max()
    if hi == lo:
        norm[:, col_i] = 0.5
        continue
    if key in ("RMSE", "MAE"):
        norm[:, col_i] = 1 - (norm[:, col_i] - lo) / (hi - lo)
    else:
        norm[:, col_i] = (norm[:, col_i] - lo) / (hi - lo)

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(norm, cmap=plt.get_cmap("YlGn"), aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_xticklabels(metrics_display, fontsize=10)
ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names, fontsize=10)
ax.set_title("Tuned Model Performance Heatmap\n(green = better, per-column normalised)",
             fontsize=13, fontweight="bold")
for i in range(len(model_names)):
    for j, key in enumerate(metric_keys):
        raw = raw_values[i, j]
        text = f"{raw:.3f}" if key == "R2" else f"{raw:.1f}"
        txt_col = "black" if norm[i, j] > 0.45 else "white"
        ax.text(j, i, text, ha="center", va="center",
                fontsize=9, color=txt_col, fontweight="bold")
plt.colorbar(im, ax=ax, label="Relative performance (1=best)", shrink=0.8)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/11_tuned_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 11_tuned_heatmap.png")

# ── Final summary table ────────────────────────────────────────────────────────
print("\n" + "="*65)
print(f"{'Model':<20} {'MAE':>8} {'RMSE':>8} {'R²':>10} {'W2%':>7} {'W5%':>7}")
print("-"*65)
for name in model_names:
    r = tuned_results[name]
    print(f"{name:<20} {r['MAE']:>8.3f} {r['RMSE']:>8.3f} "
          f"{r['R2']:>10.4f} {r['Within2']:>7.1f} {r['Within5']:>7.1f}")
print("="*65)
print(f"\nAll plots  → {PLOT_DIR}")
print(f"CV results → {cv_csv_path}")

=== Loading dataset ===
  1,485,507 rows | 19,945 journeys

=== Train/Test Split ===
  Train: 1,187,531  |  Test: 297,976

=== GridSearchCV (group-aware, 3-fold) ===

  Ridge  (2 combinations × 3-fold CV) …
    Best params : {'model__alpha': 0.001, 'model__solver': 'auto'}
    Best CV MAE : 3.6080
    Test  MAE=3.642  RMSE=7.093  R²=0.3466

  Random Forest  (1 combinations × 3-fold CV) …
    Best params : {'max_depth': 20, 'max_features': 0.5, 'n_estimators': 100}
    Best CV MAE : 3.2503
    Test  MAE=3.256  RMSE=6.688  R²=0.4191

  Gradient Boosting  (1 combinations × 3-fold CV) …
    Best params : {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}
    Best CV MAE : 3.2684
    Test  MAE=3.266  RMSE=6.620  R²=0.4309

  XGBoost  (1 combinations × 3-fold CV) …
    Best params : {'learning_rate': 0.1, 'max_depth': 8, 'subsample': 0.8}
    Best CV MAE : 3.2485
    Test  MAE=3.245  RMSE=6.562  R²=0.4409

  LightGBM  (1 combinations × 3-fold CV) …
    Best params : {'learning_rate